# Ali-CCP Dataset Exploration for Multi-Task CVR Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/rec_system_experimental/blob/main/notebooks/04_aliccp_cvr/01_data_exploration.ipynb)
[![Dataset](https://img.shields.io/badge/Dataset-Ali--CCP-blue)](https://tianchi.aliyun.com/dataset/408)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the Ali-CCP (Alibaba Click and Conversion Prediction) dataset structure and its binary-delimited format
2. Parse the multi-file data schema: sample skeletons + common features joined by hash IDs
3. Analyze the **impression → click → conversion** funnel and its extreme class imbalance
4. Understand the **Sample Selection Bias (SSB)** problem and why it motivates multi-task learning
5. Preprocess raw data into a GPU-ready format for ESMM, MMoE, and PLE models

## Prerequisites

- Python 3.8+
- PyTorch, NumPy, Pandas, Matplotlib, Seaborn, scikit-learn
- Ali-CCP sample data in `data/aliccp/`

## Table of Contents

1. [Dataset Background](#1-dataset-background)
2. [Setup & Configuration](#2-setup--configuration)
3. [Data Loading & Parsing](#3-data-loading--parsing)
4. [Basic Statistics](#4-basic-statistics)
5. [Conversion Funnel Analysis](#5-conversion-funnel-analysis)
6. [Feature Field Analysis](#6-feature-field-analysis)
7. [Sample Selection Bias (SSB)](#7-sample-selection-bias-ssb)
8. [Feature Value Distributions](#8-feature-value-distributions)
9. [Data Sparsity Analysis](#9-data-sparsity-analysis)
10. [Data Preprocessing Pipeline](#10-data-preprocessing-pipeline)
11. [Why Multi-Task Learning Solves SSB](#11-why-multi-task-learning-solves-ssb)
12. [Exercises](#exercises)
13. [Summary & Key Takeaways](#summary--key-takeaways)

## 1. Dataset Background

### Column Descriptions

The Ali-CCP dataset (released with the ESMM paper, SIGIR 2018) contains real ad impression
logs from Taobao. Features are organized into **fields** (feature groups):

#### Sample Labels (in sample_skeleton files)

| Column | Type | Description |
|--------|------|-------------|
| `sample_id` | Integer | Unique impression identifier |
| `click` | Binary (0/1) | Whether user clicked the ad |
| `conversion` | Binary (0/1) | Whether user purchased after clicking (our CVR target) |
| `hash_id` | String | Links to shared user/context features in common_features |

#### Feature Encoding Format

Features use a binary delimiter scheme:
- `\x01` separates individual features
- `\x02` separates field_id from feature data
- `\x03` separates feature_id from value

Example: `field1\x02feat1\x03val1\x01field2\x02feat2\x03val2`

#### Feature Fields

| Field ID | Category | Description | Cardinality |
|----------|----------|-------------|-------------|
| 101 | User ID | Unique user identifier | ~92K |
| 121 | User Profile | User profile segment | ~97 |
| 122 | User Group | User marketing group | ~13 |
| 124 | Gender | User gender | 2 |
| 125 | Age | User age bracket | 7 |
| 126 | Consumption Level | User spending tier | 3 |
| 127 | User Feature | Additional user segment | 3 |
| 128 | Occupation | User occupation status | 2 |
| 129 | Geography | User geographic region | 4 |
| 205 | Item ID | Unique item/ad identifier | ~970K |
| 206 | Item Category | Product category | ~7K |
| 207 | Item Feature | Additional item attribute | ~348K |
| 210 | Item Intention | Ad targeting intent | ~95K |
| 216 | Item Brand | Product brand | ~132K |
| 301 | Context | Contextual feature (e.g., ad slot) | 3 |
| 109_14 | User Categories | Categories user has browsed | ~11K |
| 110_14 | User Shops | Shops user has visited | ~1.5M |
| 127_14 | User Brands | Brands user has interacted with | ~307K |
| 150_14 | User Intentions | User inferred purchase intents | ~97K |
| 508 | User-Item Categories | Cross feature | ~6K |

#### Common Features (in common_features files)

| Column | Description |
|--------|-------------|
| `hash_id` | Join key matching sample_skeleton |
| `n_features` | Number of common features |
| `features` | Binary-encoded features (same format) |

> **Concept:** The two-file design separates **sample-specific** features (user behavior signals
> that change per impression) from **common** features (ad/item attributes shared across
> impressions with the same hash_id). This reduces storage redundancy.

### Task Definition

| Metric | Definition | Positive Rate | Challenge |
|--------|-----------|--------------|----------|
| **CTR** | P(click \| impression) | ~4.6% | Standard classification |
| **CVR** | P(conversion \| click) | ~0.6% | Only observable for clicked items (SSB!) |
| **CTCVR** | P(conversion \| impression) = CTR x CVR | ~0.03% | Extreme sparsity |

## 2. Setup & Configuration

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import os
import sys
import json
import time
import pickle
import warnings
from pathlib import Path
from collections import defaultdict, Counter, OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split

import torch

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

# Paths
DATA_DIR = Path('../../data/aliccp')
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# How many samples to load for EDA (use fewer for faster exploration)
EDA_SAMPLE_SIZE = 500_000
# Full preprocessing sample size
FULL_SAMPLE_SIZE = 5_000_000

print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Processed directory: {PROCESSED_DIR.resolve()}')
print(f'Files: {[f for f in os.listdir(DATA_DIR) if not f.endswith(".tar.gz")]}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 3. Data Loading & Parsing

> **Concept: Ali-CCP Data Format**
>
> The Ali-CCP dataset contains real-world click and conversion logs from Taobao's
> advertising system. It has two file types:
>
> - **sample_skeleton**: Per-impression records with `(sample_id, click, conversion, hash_id, n_features, features)`
> - **common_features**: Shared ad/item features keyed by `hash_id`
>
> Features are encoded with binary delimiters (`\x01`, `\x02`, `\x03`) rather than
> standard CSV format, so we need a custom parser.

In [ ]:
def parse_features(feat_bytes):
    """Parse Ali-CCP feature string with binary delimiters.
    
    Format: field_id\x02feature_id\x03value\x01field_id\x02feature_id\x03value...
    
    Args:
        feat_bytes: Raw bytes from the CSV feature column
    
    Returns:
        list of (field_id, feature_id, value) tuples
    """
    features = []
    for feat in feat_bytes.split(b'\x01'):
        feat = feat.strip()
        if not feat:
            continue
        parts = feat.split(b'\x02')
        if len(parts) != 2:
            continue
        field_id = int(parts[0])
        id_val = parts[1].split(b'\x03')
        if len(id_val) == 2:
            feat_id = int(id_val[0])
            value = float(id_val[1])
            features.append((field_id, feat_id, value))
    return features


# Quick test on a raw line
with open(DATA_DIR / 'sample_skeleton_train.csv', 'rb') as f:
    raw_line = f.readline()
parts = raw_line.split(b',', 5)
print(f'Raw line has {len(parts)} fields:')
print(f'  sample_id = {parts[0]}')
print(f'  click     = {int(parts[1])}')
print(f'  conversion= {int(parts[2])}')
print(f'  hash_id   = {parts[3][:30]}...')
print(f'  n_features= {int(parts[4])}')
sample_feats = parse_features(parts[5])
print(f'\nParsed {len(sample_feats)} features from first line')
print(f'First 5 features: (field_id, feat_id, value)')
for field, fid, val in sample_feats[:5]:
    print(f'  field={field}, feat_id={fid}, value={val}')

In [ ]:
def load_skeleton(filepath, max_samples=None):
    """Load skeleton file, return labels and per-sample features.
    
    Args:
        filepath: Path to sample_skeleton CSV
        max_samples: Maximum number of samples to load (None for all)
    
    Returns:
        clicks: numpy array of click labels
        conversions: numpy array of conversion labels
        hash_ids: list of hash ID strings
        sample_features: list of dicts {field_id: [(feat_id, value), ...]}
    """
    clicks, conversions = [], []
    hash_ids = []
    sample_features = []
    
    with open(filepath, 'rb') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            parts = line.split(b',', 5)
            clicks.append(int(parts[1]))
            conversions.append(int(parts[2]))
            hash_ids.append(parts[3].decode())
            
            feats = parse_features(parts[5])
            feat_dict = defaultdict(list)
            for field, fid, val in feats:
                feat_dict[field].append((fid, val))
            sample_features.append(dict(feat_dict))
            
            if (i + 1) % 100_000 == 0:
                print(f'  Loaded {i+1:,} samples...')
    
    return np.array(clicks), np.array(conversions), hash_ids, sample_features


# Load training data for EDA
print(f'Loading first {EDA_SAMPLE_SIZE:,} training samples for EDA...')
start = time.time()
train_clicks, train_convs, train_hashes, train_feats = load_skeleton(
    DATA_DIR / 'sample_skeleton_train.csv', max_samples=EDA_SAMPLE_SIZE
)
elapsed = time.time() - start
print(f'Loaded {len(train_clicks):,} samples in {elapsed:.1f}s')
print(f'  Clicks: {train_clicks.sum():,} ({train_clicks.mean()*100:.2f}%)')
print(f'  Conversions: {train_convs.sum():,} ({train_convs.mean()*100:.4f}%)')

In [ ]:
# Preview the data structure
print("=" * 80)
print("DATA PREVIEW")
print("=" * 80)

print(f'\nSample 0:')
print(f'  Click: {train_clicks[0]}, Conversion: {train_convs[0]}')
print(f'  Hash ID: {train_hashes[0]}')
print(f'  Feature fields present: {sorted(train_feats[0].keys())}')
for field_id in sorted(list(train_feats[0].keys()))[:5]:
    print(f'    Field {field_id}: {train_feats[0][field_id][:3]}...')

print(f'\n{"=" * 80}')
print("DATA SHAPES")
print(f'{"=" * 80}')
print(f'  clicks:    {train_clicks.shape}, dtype={train_clicks.dtype}')
print(f'  conversions: {train_convs.shape}, dtype={train_convs.dtype}')
print(f'  hash_ids:  {len(train_hashes)} strings')
print(f'  features:  {len(train_feats)} dicts')

## 4. Basic Statistics

Let's examine the fundamental properties of the dataset: sample counts, label rates,
and unique hash IDs.

In [ ]:
# Basic statistics
n_impressions = len(train_clicks)
n_clicks = int(train_clicks.sum())
n_conversions = int(train_convs.sum())
n_unique_hashes = len(set(train_hashes))

ctr = n_clicks / n_impressions
ctcvr = n_conversions / n_impressions
cvr_clicked = n_conversions / n_clicks if n_clicks > 0 else 0

print("=" * 60)
print("BASIC STATISTICS")
print("=" * 60)
print(f'Total impressions:     {n_impressions:>12,}')
print(f'Total clicks:          {n_clicks:>12,}  ({ctr*100:.2f}%)')
print(f'Total conversions:     {n_conversions:>12,}  ({ctcvr*100:.4f}%)')
print(f'Unique hash IDs:       {n_unique_hashes:>12,}')
print(f'\nConversion Rates:')
print(f'  CTR (Click/Impression):        {ctr*100:.2f}%')
print(f'  CVR (Conversion/Click):        {cvr_clicked*100:.2f}%')
print(f'  CTCVR (Conversion/Impression): {ctcvr*100:.4f}%')
print(f'\nClass Imbalance:')
print(f'  Click ratio:      1:{int(1/ctr) if ctr > 0 else "inf"}')
print(f'  Conversion ratio: 1:{int(1/ctcvr) if ctcvr > 0 else "inf"}')

## 5. Conversion Funnel Analysis

The core of multi-task prediction is understanding the **impression → click → conversion** funnel.

> **Concept:** In e-commerce advertising, a user sees an ad impression, may click it, and may
> then purchase (convert). The conversion event is analogous to the "like" action in video
> recommendation: it only happens after a click and is extremely rare. This makes it a
> perfect target for ESMM-style multi-task modeling.

In [ ]:
# Fig 1: Conversion Funnel Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Funnel bar chart
ax = axes[0]
stages = ['Impressions', 'Clicks', 'Conversions']
counts = [n_impressions, n_clicks, n_conversions]
colors = ['#4e79a7', '#f28e2b', '#e15759']

bars = ax.bar(stages, counts, color=colors, edgecolor='white', linewidth=2)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.02,
            f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Conversion Funnel (Log Scale)', fontsize=14)
ax.set_yscale('log')
ax.set_ylim(1, max(counts) * 5)

# Right: Rate waterfall
ax = axes[1]
rates = [100.0, ctr * 100, ctcvr * 100]
labels = ['All\nImpressions', 'Clicked\n(CTR)', 'Converted\n(CTCVR)']
ax.bar(labels, rates, color=colors, edgecolor='white', linewidth=2)
for i, (label, rate) in enumerate(zip(labels, rates)):
    ax.text(i, rate + max(rates)*0.02, f'{rate:.3f}%', ha='center', va='bottom',
            fontsize=11, fontweight='bold')
ax.set_ylabel('Rate (%)')
ax.set_title('Funnel Rates (% of Impressions)', fontsize=14)
ax.set_yscale('log')

plt.suptitle('Ali-CCP: Impression \u2192 Click \u2192 Conversion Funnel', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f'Funnel Drop-off:')
print(f'  Impression \u2192 Click:      {(1 - ctr)*100:.1f}% lost')
print(f'  Click \u2192 Conversion:      {(1 - cvr_clicked)*100:.1f}% lost')
print(f'  Overall Conversion Rate:  {ctcvr*100:.4f}%')

In [ ]:
# Fig 2: Label distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Click distribution
ax = axes[0]
click_counts = [n_impressions - n_clicks, n_clicks]
ax.bar(['No Click (0)', 'Click (1)'], click_counts, color=['#a0cbe8', '#f28e2b'])
for i, c in enumerate(click_counts):
    ax.text(i, c + max(click_counts)*0.02, f'{c:,}\n({c/n_impressions*100:.1f}%)',
            ha='center', fontsize=10)
ax.set_title('Click Label Distribution', fontsize=13)
ax.set_ylabel('Count')

# Conversion distribution (all samples)
ax = axes[1]
conv_counts = [n_impressions - n_conversions, n_conversions]
ax.bar(['No Conv (0)', 'Conversion (1)'], conv_counts, color=['#a0cbe8', '#e15759'])
for i, c in enumerate(conv_counts):
    ax.text(i, c + max(conv_counts)*0.02, f'{c:,}\n({c/n_impressions*100:.2f}%)',
            ha='center', fontsize=10)
ax.set_title('Conversion Label (All Impressions)', fontsize=13)
ax.set_ylabel('Count')

# Conversion distribution (clicked only)
ax = axes[2]
clicked_convs = int(train_convs[train_clicks == 1].sum())
clicked_no_conv = n_clicks - clicked_convs
ax.bar(['No Conv (0)', 'Conversion (1)'], [clicked_no_conv, clicked_convs],
       color=['#a0cbe8', '#e15759'])
for i, c in enumerate([clicked_no_conv, clicked_convs]):
    pct = c / n_clicks * 100 if n_clicks > 0 else 0
    ax.text(i, c + max(clicked_no_conv, clicked_convs)*0.02,
            f'{c:,}\n({pct:.2f}%)', ha='center', fontsize=10)
ax.set_title('Conversion Label (Clicked Only)', fontsize=13)
ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Fig 3: Conditional conversion rate comparison
fig, ax = plt.subplots(figsize=(10, 6))

cvr_from_clicked = n_conversions / n_clicks if n_clicks > 0 else 0
cvr_from_all = n_conversions / n_impressions

bars = ax.bar(
    ['P(conv | click)\n(CVR on Clicked)', 'P(conv | impression)\n(CTCVR on All)'],
    [cvr_from_clicked * 100, cvr_from_all * 100],
    color=['#f28e2b', '#e15759'], edgecolor='white', linewidth=2, width=0.5
)
for bar, val in zip(bars, [cvr_from_clicked, cvr_from_all]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val*100:.3f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylabel('Rate (%)', fontsize=12)
ax.set_title('Conversion Rate: Clicked Samples vs All Impressions', fontsize=14)
ax.axhline(y=cvr_from_all * 100, color='red', linestyle='--', alpha=0.5, label='True CTCVR')
ax.legend()
plt.tight_layout()
plt.show()

ratio = cvr_from_clicked / cvr_from_all if cvr_from_all > 0 else float('inf')
print(f'CVR gap: P(conv|click) = {cvr_from_clicked*100:.3f}% vs '
      f'P(conv|impression) = {cvr_from_all*100:.4f}%')
print(f'Ratio: {ratio:.1f}x higher in clicked population')

## 6. Feature Field Analysis

The Ali-CCP dataset organizes features into **fields** (groups). Each field represents a
different type of information: user profile, item attributes, context, etc.

> **Pro Tip:** Understanding the feature fields is crucial for model design. Some fields
> (like user ID and item ID) are high-cardinality and need larger embedding dimensions,
> while context features (like ad slot) have low cardinality.

In [ ]:
# Feature field analysis
field_counter = Counter()
field_unique_feats = defaultdict(set)

n_analyze = min(500_000, len(train_feats))
for feat_dict in train_feats[:n_analyze]:
    for field_id, feat_list in feat_dict.items():
        field_counter[field_id] += len(feat_list)
        for fid, val in feat_list:
            field_unique_feats[field_id].add(fid)

print("=" * 70)
print("FEATURE FIELD ANALYSIS")
print("=" * 70)
print(f'{"Field ID":>10} {"Occurrences":>15} {"Unique Features":>18} {"Category":>15}')
print('-' * 60)

field_categories = {
    101: 'User', 109: 'User Hist', 110: 'User Hist',
    121: 'User', 122: 'User', 124: 'User', 125: 'User',
    126: 'User', 127: 'User', 128: 'User', 129: 'User',
    150: 'User Hist',
    205: 'Item', 206: 'Item', 207: 'Item',
    210: 'Item', 216: 'Item',
    301: 'Context',
    508: 'Cross', 509: 'Cross', 702: 'Cross', 853: 'Cross',
}

for field_id in sorted(field_counter.keys()):
    cat = field_categories.get(field_id, 'Other')
    print(f'{field_id:>10} {field_counter[field_id]:>15,} '
          f'{len(field_unique_feats[field_id]):>18,} {cat:>15}')

In [ ]:
# Fig 4: Feature field visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Field cardinality (unique features per field)
ax = axes[0]
fields = sorted(field_unique_feats.keys())
cardinalities = [len(field_unique_feats[f]) for f in fields]
field_labels = [str(f) for f in fields]

cat_colors = {'User': '#4e79a7', 'User Hist': '#76b7b2', 'Item': '#f28e2b',
              'Context': '#59a14f', 'Cross': '#e15759', 'Other': '#bab0ac'}
bar_colors = [cat_colors.get(field_categories.get(f, 'Other'), '#bab0ac') for f in fields]

ax.barh(field_labels, cardinalities, color=bar_colors, edgecolor='white')
ax.set_xlabel('Number of Unique Features (log scale)')
ax.set_ylabel('Field ID')
ax.set_title('Feature Cardinality per Field', fontsize=13)
ax.set_xscale('log')

handles = [mpatches.Patch(color=c, label=l) for l, c in cat_colors.items()]
ax.legend(handles=handles, loc='lower right', fontsize=9)

# Field occurrence frequency
ax = axes[1]
occurrences = [field_counter[f] for f in fields]
ax.barh(field_labels, occurrences, color=bar_colors, edgecolor='white')
ax.set_xlabel('Total Feature Occurrences (log scale)')
ax.set_ylabel('Field ID')
ax.set_title('Feature Occurrence per Field', fontsize=13)
ax.set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Fig 5: Number of features per sample
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fields per sample
ax = axes[0]
fields_per_sample = [len(fd) for fd in train_feats]
ax.hist(fields_per_sample, bins=range(0, max(fields_per_sample) + 2),
        color='#4e79a7', edgecolor='white', alpha=0.8)
ax.set_xlabel('Number of Feature Fields')
ax.set_ylabel('Count')
ax.set_title('Feature Fields per Sample', fontsize=13)
ax.axvline(np.mean(fields_per_sample), color='red', linestyle='--',
           label=f'Mean: {np.mean(fields_per_sample):.1f}')
ax.legend()

# Total features per sample (including multi-valued)
ax = axes[1]
total_feats_per_sample = [sum(len(v) for v in fd.values()) for fd in train_feats]
ax.hist(total_feats_per_sample, bins=50, color='#f28e2b', edgecolor='white', alpha=0.8)
ax.set_xlabel('Total Number of Features')
ax.set_ylabel('Count')
ax.set_title('Total Features per Sample (incl. multi-valued)', fontsize=13)
ax.axvline(np.mean(total_feats_per_sample), color='red', linestyle='--',
           label=f'Mean: {np.mean(total_feats_per_sample):.1f}')
ax.legend()

plt.tight_layout()
plt.show()

## 7. Sample Selection Bias (SSB)

This is the **central challenge** that motivates multi-task learning for conversion prediction.

> **Concept: Sample Selection Bias (SSB)**
>
> We want to estimate $P(\text{conversion} | \text{impression})$ for all ad impressions.
>
> However, we only observe conversion labels for **clicked** impressions:
> - If a user did not click, we don't know if they would have converted
> - A naive CVR model trains only on clicked samples: $P(\text{conversion} | \text{click})$
> - But at serving time, we must predict for ALL impressions (including unclicked)
>
> This creates a **distribution mismatch** between training and inference.
>
> **Mathematical formulation:**
>
> $$P_{\text{train}}(x) = P(x | \text{click} = 1) \neq P_{\text{serve}}(x) = P(x)$$
>
> The training distribution is a biased subset of the serving distribution.

In [ ]:
# Fig 6: Sample Selection Bias Illustration
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: The bias diagram
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('Sample Selection Bias in CVR Prediction', fontsize=14)

# All impressions box
ax.add_patch(plt.Rectangle((0.5, 0.5), 9, 6, fill=True, facecolor='#e8f4fd',
             edgecolor='#4e79a7', linewidth=2))
ax.text(5, 6.2, f'All Impressions: {n_impressions:,}', ha='center', fontsize=12,
        fontweight='bold', color='#4e79a7')

# Clicked box (subset)
ax.add_patch(plt.Rectangle((1.5, 1), 7, 4, fill=True, facecolor='#fff2e0',
             edgecolor='#f28e2b', linewidth=2))
ax.text(5, 4.7, f'Clicked: {n_clicks:,} ({ctr*100:.1f}%)', ha='center', fontsize=11,
        fontweight='bold', color='#f28e2b')

# Converted box
ax.add_patch(plt.Rectangle((3, 1.5), 4, 2.5, fill=True, facecolor='#fde0e0',
             edgecolor='#e15759', linewidth=2))
ax.text(5, 3.5, f'Converted: {n_conversions:,}', ha='center', fontsize=11,
        fontweight='bold', color='#e15759')
ax.text(5, 2.8, f'({ctcvr*100:.4f}% of all)', ha='center', fontsize=10, color='#e15759')

# Annotations
ax.annotate('Naive CVR trains HERE\n(biased subset)',
            xy=(5, 1), xytext=(5, 0.2), ha='center',
            fontsize=10, color='#e15759',
            arrowprops=dict(arrowstyle='->', color='#e15759'))

# Right: Quantification
ax = axes[1]
categories = ['CVR\n(Clicked Only)', 'CTCVR\n(All Impressions)']
values = [cvr_from_clicked * 100, cvr_from_all * 100]
bar_colors_ssb = ['#f28e2b', '#e15759']

bars = ax.bar(categories, values, color=bar_colors_ssb, edgecolor='white', linewidth=2, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
            f'{val:.3f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('SSB Quantification: Training vs Serving Gap', fontsize=14)

if values[0] > 0 and values[1] > 0:
    ax.annotate(f'{values[0]/values[1]:.0f}x overestimate',
                xy=(0.5, (values[0]+values[1])/2), fontsize=11,
                ha='center', color='red', fontweight='bold')

plt.tight_layout()
plt.show()

> **Common Pitfall:** Training a conversion prediction model only on clicked samples produces
> a model that systematically **overestimates** conversion rates. This is because the clicked
> population is a biased sample -- users who clicked already showed purchase intent. The model
> learns $P(\text{conversion}|\text{click})$ but is applied to predict
> $P(\text{conversion}|\text{impression})$, leading to inflated predictions.

In [ ]:
# Quantify the bias: compare feature distributions between clicked and unclicked
print("=" * 60)
print("SAMPLE SELECTION BIAS QUANTIFICATION")
print("=" * 60)

clicked_mask = train_clicks == 1
unclicked_mask = train_clicks == 0

print(f'\nClicked samples:   {clicked_mask.sum():>10,} ({clicked_mask.mean()*100:.1f}%)')
print(f'Unclicked samples: {unclicked_mask.sum():>10,} ({unclicked_mask.mean()*100:.1f}%)')

# Compare feature field presence between clicked and unclicked
clicked_fields = Counter()
unclicked_fields = Counter()

n_sub = min(200_000, len(train_feats))
for i, fd in enumerate(train_feats[:n_sub]):
    if train_clicks[i] == 1:
        for field in fd:
            clicked_fields[field] += 1
    else:
        for field in fd:
            unclicked_fields[field] += 1

n_clicked_sub = sum(1 for c in train_clicks[:n_sub] if c == 1)
n_unclicked_sub = n_sub - n_clicked_sub

print(f'\n{"Field":>8} {"Clicked %":>12} {"Unclicked %":>12} {"Diff":>8}')
print('-' * 45)
for field in sorted(set(list(clicked_fields.keys()) + list(unclicked_fields.keys()))):
    c_pct = clicked_fields[field] / max(n_clicked_sub, 1) * 100
    u_pct = unclicked_fields[field] / max(n_unclicked_sub, 1) * 100
    diff = abs(c_pct - u_pct)
    marker = ' ***' if diff > 5 else ''
    print(f'{field:>8} {c_pct:>11.1f}% {u_pct:>11.1f}% {diff:>7.1f}%{marker}')

## 8. Feature Value Distributions

Let's examine how feature values are distributed within each field, and whether
they differ between clicked and unclicked samples.

In [ ]:
# Fig 7: Feature value distributions for top fields
field_values = defaultdict(list)

for i, fd in enumerate(train_feats[:min(100_000, len(train_feats))]):
    for field_id, feat_list in fd.items():
        for fid, val in feat_list:
            field_values[field_id].append(val)

# Plot distributions for first 8 fields
interesting_fields = sorted(field_values.keys())[:8]
n_fields = len(interesting_fields)
n_cols = 4
n_rows = (n_fields + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for idx, field_id in enumerate(interesting_fields):
    r, c = idx // n_cols, idx % n_cols
    ax = axes[r][c]
    vals = np.array(field_values[field_id])
    unique_vals = np.unique(vals)
    
    if len(unique_vals) <= 20:
        ax.hist(vals, bins=len(unique_vals), color='#4e79a7', edgecolor='white', alpha=0.8)
    else:
        ax.hist(vals, bins=50, color='#4e79a7', edgecolor='white', alpha=0.8)
    
    cat = field_categories.get(field_id, 'Other')
    ax.set_title(f'Field {field_id} ({cat})', fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

# Hide extra axes
for idx in range(n_fields, n_rows * n_cols):
    r, c = idx // n_cols, idx % n_cols
    axes[r][c].set_visible(False)

plt.suptitle('Feature Value Distributions by Field', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Data Sparsity Analysis

> **Concept:** In large-scale ad datasets, most feature IDs appear very rarely. This
> long-tail distribution makes embedding-based approaches essential -- we can't use
> one-hot encoding when feature cardinalities reach millions.

In [ ]:
# Fig 8: Data sparsity analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature frequency distribution
ax = axes[0]
all_feat_freqs = Counter()
for fd in train_feats[:min(200_000, len(train_feats))]:
    for field_id, feat_list in fd.items():
        for fid, val in feat_list:
            all_feat_freqs[(field_id, fid)] += 1

freq_values = list(all_feat_freqs.values())
ax.hist(freq_values, bins=100, color='#4e79a7', edgecolor='white', alpha=0.8)
ax.set_xlabel('Feature Frequency')
ax.set_ylabel('Number of Features')
ax.set_title('Feature Frequency Distribution (Long Tail)', fontsize=13)
ax.set_yscale('log')
ax.axvline(np.median(freq_values), color='red', linestyle='--',
           label=f'Median: {np.median(freq_values):.0f}')
ax.legend()

# Hash ID reuse distribution
ax = axes[1]
hash_counts = Counter(train_hashes)
reuse_values = list(hash_counts.values())
ax.hist(reuse_values, bins=50, color='#f28e2b', edgecolor='white', alpha=0.8)
ax.set_xlabel('Number of Impressions per Hash ID')
ax.set_ylabel('Number of Hash IDs')
ax.set_title('Hash ID Reuse Distribution', fontsize=13)
ax.set_yscale('log')
ax.axvline(np.median(reuse_values), color='red', linestyle='--',
           label=f'Median: {np.median(reuse_values):.0f}')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Total unique (field, feature) pairs: {len(all_feat_freqs):,}')
print(f'Features appearing only once: {sum(1 for v in freq_values if v == 1):,} '
      f'({sum(1 for v in freq_values if v == 1)/len(freq_values)*100:.1f}%)')
print(f'Unique hash IDs: {len(hash_counts):,}')
print(f'Hash IDs appearing once: {sum(1 for v in reuse_values if v == 1):,}')

## 10. Data Preprocessing Pipeline

Prepare the data for multi-task learning models. We need:
1. Load the full training set (up to 5M samples)
2. Load and join common features by hash ID
3. Build per-field feature ID mappings (contiguous integers starting from 1, with 0 as padding)
4. Encode each sample into a fixed-size feature vector
5. Train/validation split (90/10)
6. Save as pickle for fast reloading

> **Pro Tip:** The encoding uses `padding_idx=0` in embeddings. Feature IDs start from 1 so
> that missing fields get the zero embedding. This is important for fields that don't appear
> in every sample.

In [ ]:
# Check if preprocessed data already exists
processed_path = PROCESSED_DIR / 'aliccp_processed.pkl'

if processed_path.exists():
    print(f'Preprocessed data already exists at {processed_path}')
    print('Loading...')
    with open(processed_path, 'rb') as f:
        data = pickle.load(f)
    print(f'Loaded. Keys: {list(data.keys())}')
    print(f'Train shape: {data["X_ids_train"].shape}')
    print(f'Val shape: {data["X_ids_val"].shape}')
    print(f'Fields: {data["all_fields"]}')
    print(f'Field cardinalities: {data["field_cardinalities"]}')
else:
    print('Preprocessed data not found. Running full pipeline below...')
    print('(This may take several minutes for 5M samples)')

In [ ]:
if not processed_path.exists():
    # Step 1: Load full training data
    print(f'Loading full training data ({FULL_SAMPLE_SIZE/1e6:.0f}M samples)...')
    start = time.time()
    clicks, convs, hash_ids, sample_features = load_skeleton(
        DATA_DIR / 'sample_skeleton_train.csv', max_samples=FULL_SAMPLE_SIZE
    )
    print(f'Loaded {len(clicks):,} samples in {time.time()-start:.1f}s')
    print(f'CTR: {clicks.mean()*100:.2f}%, CTCVR: {convs.mean()*100:.4f}%')

In [ ]:
if not processed_path.exists():
    # Step 2: Load common features
    print('\nLoading common features...')
    start = time.time()
    needed = set(hash_ids)
    common_features = {}
    
    with open(DATA_DIR / 'common_features_train.csv', 'rb') as f:
        for i, line in enumerate(f):
            parts = line.split(b',', 2)
            h = parts[0].decode()
            if h in needed:
                feats = parse_features(parts[2])
                fd = defaultdict(list)
                for field, fid, val in feats:
                    fd[field].append((fid, val))
                common_features[h] = dict(fd)
            if (i + 1) % 50_000 == 0:
                print(f'  Scanned {i+1:,}, matched {len(common_features):,}')
    
    print(f'Loaded {len(common_features):,} common features in {time.time()-start:.1f}s')

In [ ]:
if not processed_path.exists():
    # Step 3: Build feature mappings
    print('\nBuilding feature mappings...')
    field_feat_ids = defaultdict(set)
    
    for fd in sample_features:
        for field, fl in fd.items():
            for fid, _ in fl:
                field_feat_ids[field].add(fid)
    for fd in common_features.values():
        for field, fl in fd.items():
            for fid, _ in fl:
                field_feat_ids[field].add(fid)
    
    feat_mappings = {}
    for field in sorted(field_feat_ids.keys()):
        mapping = {fid: idx + 1 for idx, fid in enumerate(sorted(field_feat_ids[field]))}
        feat_mappings[field] = mapping
        print(f'  Field {field}: {len(mapping):,} features')
    
    all_fields = sorted(field_feat_ids.keys())
    print(f'\nTotal fields: {len(all_fields)}')
    print(f'Total unique features: {sum(len(m) for m in feat_mappings.values()):,}')

In [ ]:
if not processed_path.exists():
    # Step 4: Encode samples into fixed-size matrices
    print('\nEncoding samples...')
    start = time.time()
    X_ids = np.zeros((len(clicks), len(all_fields)), dtype=np.int64)
    X_vals = np.zeros((len(clicks), len(all_fields)), dtype=np.float32)
    
    for i in range(len(clicks)):
        merged = {}
        c = common_features.get(hash_ids[i], {})
        if c:
            merged.update(c)
        merged.update(sample_features[i])  # sample features override common
        
        for j, field in enumerate(all_fields):
            if field in merged and merged[field]:
                fid, val = merged[field][0]  # take first feature per field
                if fid in feat_mappings[field]:
                    X_ids[i, j] = feat_mappings[field][fid]
                    X_vals[i, j] = val
        
        if (i + 1) % 1_000_000 == 0:
            print(f'  Encoded {i+1:,}...')
    
    print(f'Encoded in {time.time()-start:.1f}s')
    print(f'X_ids shape: {X_ids.shape}, X_vals shape: {X_vals.shape}')
    print(f'Feature coverage: {(X_ids > 0).mean()*100:.1f}% non-zero entries')

In [ ]:
if not processed_path.exists():
    # Step 5: Train/validation split
    indices = np.arange(len(clicks))
    train_idx, val_idx = train_test_split(indices, test_size=0.1, random_state=42)
    
    data = {
        'X_ids_train': X_ids[train_idx], 'X_vals_train': X_vals[train_idx],
        'y_click_train': clicks[train_idx].astype(np.float32),
        'y_conv_train': convs[train_idx].astype(np.float32),
        'X_ids_val': X_ids[val_idx], 'X_vals_val': X_vals[val_idx],
        'y_click_val': clicks[val_idx].astype(np.float32),
        'y_conv_val': convs[val_idx].astype(np.float32),
        'all_fields': all_fields,
        'feat_mappings': feat_mappings,
        'field_cardinalities': {f: len(m) + 1 for f, m in feat_mappings.items()},
    }
    
    # Step 6: Save
    with open(processed_path, 'wb') as f:
        pickle.dump(data, f)
    
    print(f'Saved preprocessed data to {processed_path}')
    print(f'Train: {len(train_idx):,}, Val: {len(val_idx):,}')
    print(f'Train CTR: {data["y_click_train"].mean()*100:.2f}%')
    print(f'Train CTCVR: {data["y_conv_train"].mean()*100:.4f}%')

In [ ]:
# Summary of preprocessed data
print("=" * 60)
print("PREPROCESSED DATA SUMMARY")
print("=" * 60)
print(f'Training samples:   {data["X_ids_train"].shape[0]:>12,}')
print(f'Validation samples: {data["X_ids_val"].shape[0]:>12,}')
print(f'Feature fields:     {len(data["all_fields"]):>12}')
print(f'\nField cardinalities (incl. padding):')
for field, card in data['field_cardinalities'].items():
    print(f'  Field {field}: {card:,}')
print(f'\nLabel distributions:')
print(f'  Train CTR:   {data["y_click_train"].mean()*100:.2f}%')
print(f'  Train CTCVR: {data["y_conv_train"].mean()*100:.4f}%')
print(f'  Val CTR:     {data["y_click_val"].mean()*100:.2f}%')
print(f'  Val CTCVR:   {data["y_conv_val"].mean()*100:.4f}%')

## 11. Why Multi-Task Learning Solves SSB

> **Concept: ESMM's Solution to Sample Selection Bias**
>
> The key insight of ESMM (Entire Space Multi-Task Model) is to decompose the prediction as:
>
> $$P(\text{conversion} | \text{impression}) = P(\text{click} | \text{impression}) \times P(\text{conversion} | \text{click}, \text{impression})$$
>
> $$\boxed{\text{CTCVR} = \text{CTR} \times \text{CVR}}$$
>
> **Why this works:**
> 1. The CTR task trains on ALL impressions (no bias)
> 2. The CVR task's output is never directly supervised with biased labels
> 3. Instead, CTCVR = CTR x CVR is supervised against the conversion label on ALL impressions
> 4. Shared embeddings between CTR and CVR towers transfer knowledge from the data-rich CTR task
>
> **Key advantage for Ali-CCP:**
> The Ali-CCP dataset was specifically designed to benchmark this approach. With CTR at ~4-5%
> and CTCVR at ~0.03%, the extreme imbalance makes SSB correction even more critical than
> in video recommendation datasets.

---

## Exercises

### Exercise 1: Conversion Rate by Feature Field
For each feature field, compute the average conversion rate when the field is present vs absent.
Which fields are most predictive of conversion?

In [ ]:
# TODO: Exercise 1
# Hint: For each field, split samples by whether the field exists in feat_dict
# Compare conversion rates in each group
# Use train_feats, train_convs

# Your code here:
# for field_id in sorted(field_unique_feats.keys()):
#     present = [i for i, fd in enumerate(train_feats) if field_id in fd]
#     absent = [i for i, fd in enumerate(train_feats) if field_id not in fd]
#     conv_present = train_convs[present].mean()
#     conv_absent = train_convs[absent].mean() if absent else 0
#     print(f'Field {field_id}: present={conv_present:.5f}, absent={conv_absent:.5f}')

pass

### Exercise 2: ESMM Decomposition Verification
Verify the ESMM decomposition holds in the data:
1. Compute CTR = P(click | impression)
2. Compute CVR = P(conversion | click)
3. Compute CTCVR = P(conversion | impression)
4. Verify: CTCVR = CTR x CVR (within floating point precision)

In [ ]:
# TODO: Exercise 2
# Hint: Use train_clicks and train_convs arrays
# ctr = train_clicks.mean()
# cvr = train_convs[train_clicks == 1].mean()
# ctcvr = train_convs.mean()
# Verify: abs(ctcvr - ctr * cvr) < 1e-10

# Your code here:
pass

### Exercise 3: Hash ID Sharing Analysis
Analyze how many impressions share the same hash_id. Do shared hash_ids have different
click/conversion rates than unique ones?

In [ ]:
# TODO: Exercise 3
# Hint: Use Counter(train_hashes) to find shared vs unique hash_ids
# Compare avg click/conversion rates for samples with shared vs unique hash_ids

# Your code here:
pass

### Exercise 4: SSB Simulation
Simulate the effect of sample selection bias by:
1. Training a simple logistic regression on clicked-only samples to predict conversion
2. Training the same model on ALL samples to predict conversion
3. Compare their AUC on all test impressions

In [ ]:
# TODO: Exercise 4
# Hint: Use sklearn.linear_model.LogisticRegression
# You'll need to encode features into a simple matrix first
# Model 1: fit on samples where click==1, predict conversion
# Model 2: fit on ALL samples, predict conversion
# Evaluate both on validation set (all samples) using roc_auc_score

# Your code here:
pass

---

## Summary & Key Takeaways

### What We Learned

1. **Ali-CCP Dataset Structure**: The dataset contains real ad impression logs from Taobao with a binary delimiter format. Features are split across sample-specific and common (ad-level) files joined by hash IDs. There are 20+ feature fields covering user demographics, item attributes, context signals, and cross features.

2. **Extreme Class Imbalance**: CTR is ~4-5% and CTCVR is ~0.03%, making conversion prediction extremely challenging. The conversion-to-impression ratio is approximately 1:3000.

3. **Sample Selection Bias (SSB)**: The conversion rate estimated from clicked samples (~0.5-0.6%) is vastly higher than the true rate on all impressions (~0.03%). A naive model trained on clicks systematically overestimates conversion probability.

4. **Feature Space Characteristics**: High-cardinality fields (millions of unique features in user behavioral history) combined with sparse presence patterns require embedding-based approaches with padding for missing fields.

5. **ESMM Motivation**: The CTCVR = CTR x CVR decomposition trains on all impressions, eliminating SSB. This is exactly what the Ali-CCP dataset was designed to benchmark.

### Next Steps

In the following notebooks, we will:
- **Notebook 02**: Implement ESMM and a naive CVR baseline from scratch in PyTorch
- **Notebook 03**: Build advanced architectures (MMoE and PLE) for multi-task CVR
- **Notebook 04**: Compare all models head-to-head with statistical analysis